# triton-blackhole — Colab smoke test

1. **Runtime → Change runtime type → GPU (T4)**
2. Run all cells.

Repo: https://github.com/brian-mwirigi/triton-blackhole

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"

In [ ]:
!pip -q install -U triton
!pip -q install "git+https://github.com/brian-mwirigi/triton-blackhole.git"
import triton, triton_blackhole as tb
print("triton", triton.__version__)
print("triton-blackhole", tb.__version__)

In [ ]:
# CPU-side debugger demos (no custom kernel needed)
!python -c "import triton_blackhole, pathlib, runpy; import triton_blackhole as t; p=pathlib.Path(t.__file__).resolve().parents[1]/examples"; print('skip local demos if site-packages');"
from triton_blackhole import compare, bisect_axes, classify_drift, format_report
from triton_blackhole.classify import format_classification
import torch

torch.manual_seed(0)
ref = torch.zeros(64, 64, device="cuda")
act = ref.clone()
act[50, 51] = 10
print(format_report(compare(act, ref, atol=1e-6, rtol=1e-6)))
print(format_classification(classify_drift(act, ref, atol=1e-6, rtol=1e-6)))
print(bisect_axes(act, ref, atol=1e-6, rtol=1e-6).report())

In [ ]:
# Live Triton kernel vs torch reference
import triton
import triton.language as tl
import torch
from triton_blackhole.triton_hooks import diagnose

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

def triton_add(x, y):
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK = 256
    add_kernel[(triton.cdiv(n, BLOCK),)](x, y, out, n, BLOCK=BLOCK)
    return out

x = torch.randn(100_000, device="cuda", dtype=torch.bfloat16)
y = torch.randn_like(x)
actual = triton_add(x, y)
reference = x + y
print(diagnose(actual, reference))
print("LIVE TRITON KERNEL OK")